# Phase 2: Data Preparation for ML Models

This notebook generates training data by:
1. Loading synthetic investor data
2. Running the existing rule-based allocation logic
3. Extracting features (X) and targets (y1, y2, y3)
4. Encoding and scaling features
5. Splitting into train/test sets
6. Saving prepared data for model training

## Step 1: Setup and Imports

In [11]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path to import allocations_temp
sys.path.append('..')
import allocations_temp

print("✅ Imports successful")

✅ Imports successful


## Step 2: Load Investor Data

In [12]:
# Load the newly generated balanced investor data
investor_file = '../synthetic_investor_data_20260104180905.csv'
investors_df = pd.read_csv(investor_file)

print(f"📊 Loaded {len(investors_df)} investors")
print(f"\nColumns: {investors_df.columns.tolist()}")
print(f"\nShape: {investors_df.shape}")
print(f"\n{'='*60}")
print("First 5 rows:")
investors_df.head()

📊 Loaded 5000 investors

Columns: ['Age', 'Gender', 'Education', 'MaritalStatus', 'HouseholdSize', 'Income', 'InvestmentGoal', 'InvestmentHorizon', 'InvestmentCapital', 'RiskTolerance', 'FinancialInvolvement', 'ExperienceYears']

Shape: (5000, 12)

First 5 rows:


,Age,Gender,Education,MaritalStatus,HouseholdSize,Income,InvestmentGoal,InvestmentHorizon,InvestmentCapital,RiskTolerance,FinancialInvolvement,ExperienceYears
0,49,Female,Master,Married,3,106525,Child education,9,102721,High,High,16
1,41,Male,High school,Married,2,84478,Retirement,5,57480,Medium,Medium,0
2,45,Male,High school,Married,4,52490,Child education,15,29231,Low,Medium,21
3,38,Female,Bachelor,Married,3,77344,Home purchase,7,61371,Medium,Medium,3
4,20,Female,Bachelor,Single,2,37339,Other,1,2562,Medium,Medium,0


In [13]:
# Verify balanced risk tolerance distribution
print("Risk Tolerance Distribution:")
print(investors_df['RiskTolerance'].value_counts())
print("\nPercentages:")
print(investors_df['RiskTolerance'].value_counts(normalize=True) * 100)

Risk Tolerance Distribution:
RiskTolerance
High      1667
Low       1667
Medium    1666
Name: count, dtype: int64

Percentages:
RiskTolerance
High      33.34
Low       33.34
Medium    33.32
Name: proportion, dtype: float64


## Step 3: Load ETF Data (Cached)

In [14]:
# Use cached ETF data to avoid API calls
etf_file = '../combined_etf_with_morning_star.csv'
etf_df = pd.read_csv(etf_file)

print(f"📊 Loaded {len(etf_df)} ETFs")
print(f"\nColumns: {etf_df.columns.tolist()[:10]}...")  # Show first 10 columns
print(f"\nShape: {etf_df.shape}")
print(f"\n{'='*60}")
print("First 3 rows (selected columns):")
etf_df[['Fund Symbol', 'Category', 'Assets Under Management (AUM)', 'custom_star_rating', 'Volatility (Annual STD)']].head(3)

📊 Loaded 95 ETFs

Columns: ['Fund Symbol', 'Fund Name', 'Fund Manager', 'Fifty Two Week High', 'Fifty Two Week Low', 'Volume', 'Average Trading Volume', 'Market Cap', 'Category', 'Assets Under Management (AUM)']...

Shape: (95, 46)

First 3 rows (selected columns):


,Fund Symbol,Category,Assets Under Management (AUM),custom_star_rating,Volatility (Annual STD)
0,VOO,Large Blend,1.409023e+12,4.0,0.181194
1,IVV,Large Blend,7.013694e+11,4.0,0.181645
2,SPY,Large Blend,6.727266e+11,4.0,0.180484


## Step 4: Run Allocation Pipeline to Generate Ground Truth Labels

In [15]:
print("🔄 Running allocation pipeline to generate ground truth labels...")
print("This will take a moment...\n")

# Run the existing rule-based logic
result_df = allocations_temp.run_allocations_pipeline(etf_df, investors_df)

print(f"\n✅ Pipeline complete!")
print(f"\nResult shape: {result_df.shape}")
print(f"\nNew columns added: {[col for col in result_df.columns if col not in investors_df.columns]}")

🔄 Running allocation pipeline to generate ground truth labels...
This will take a moment...

... [Allocations] Preparing ETF Dictionary...
    -> Found 70 Equity ETFs
    -> Found 21 Bond ETFs
    -> Found 4 Alternative ETFs
... [Allocations] Processing 5000 Investors...
... [Allocations] Assigning Specific ETFs...

✅ Pipeline complete!

Result shape: (5000, 20)

New columns added: ['age', 'income', 'horizon', 'risk_tolerance', 'experience', 'profile_probs', 'risk_profile', 'allocation', 'num_equity_etfs', 'num_bond_etfs', 'num_alt_etfs', 'total_etfs', 'portfolio_etfs']


In [16]:
# Inspect the results
print("All columns in result_df:")
print(result_df.columns.tolist())

All columns in result_df:
['age', 'Gender', 'Education', 'MaritalStatus', 'HouseholdSize', 'income', 'InvestmentGoal', 'horizon', 'InvestmentCapital', 'risk_tolerance', 'FinancialInvolvement', 'experience', 'profile_probs', 'risk_profile', 'allocation', 'num_equity_etfs', 'num_bond_etfs', 'num_alt_etfs', 'total_etfs', 'portfolio_etfs']


In [17]:
# Check risk profile distribution (from pipeline)
print("Risk Profile Distribution (from pipeline):")
print(result_df['risk_profile'].value_counts())
print("\nPercentages:")
print(result_df['risk_profile'].value_counts(normalize=True) * 100)

Risk Profile Distribution (from pipeline):
risk_profile
Moderate        1901
Aggressive      1571
Conservative    1528
Name: count, dtype: int64

Percentages:
risk_profile
Moderate        38.02
Aggressive      31.42
Conservative    30.56
Name: proportion, dtype: float64


## Step 5: Extract Features (X)

In [20]:
# Define feature columns (using ACTUAL column names from result_df)
feature_columns = [
    'age',                    # lowercase
    'Gender',
    'Education',
    'MaritalStatus',
    'HouseholdSize',
    'income',                 # lowercase
    'InvestmentGoal',
    'horizon',                # was InvestmentHorizon
    'InvestmentCapital',
    'risk_tolerance',         # was RiskTolerance
    'FinancialInvolvement',
    'experience'              # was ExperienceYears
]

X = result_df[feature_columns].copy()

print(f"✅ Extracted features (X)")
print(f"Shape: {X.shape}")
print(f"\nFeature types:")
print(X.dtypes)
print(f"\n{'='*60}")
print("First 5 rows:")
X.head()

✅ Extracted features (X)
Shape: (5000, 12)

Feature types:
age                      int64
Gender                  object
Education               object
MaritalStatus           object
HouseholdSize            int64
income                   int64
InvestmentGoal          object
horizon                  int64
InvestmentCapital        int64
risk_tolerance          object
FinancialInvolvement    object
experience               int64
dtype: object

First 5 rows:


,age,Gender,Education,MaritalStatus,HouseholdSize,income,InvestmentGoal,horizon,InvestmentCapital,risk_tolerance,FinancialInvolvement,experience
0,49,Female,Master,Married,3,106525,Child education,9,102721,High,High,16
1,41,Male,High school,Married,2,84478,Retirement,5,57480,Medium,Medium,0
2,45,Male,High school,Married,4,52490,Child education,15,29231,Low,Medium,21
3,38,Female,Bachelor,Married,3,77344,Home purchase,7,61371,Medium,Medium,3
4,20,Female,Bachelor,Single,2,37339,Other,1,2562,Medium,Medium,0


## Step 6: Extract Targets (y1, y2, y3)

In [21]:
# Target 1: Risk Profile (Classification)
y1_risk_profile = result_df['risk_profile'].copy()

print("Target 1: Risk Profile (Classification)")
print(f"Shape: {y1_risk_profile.shape}")
print(f"\nDistribution:")
print(y1_risk_profile.value_counts())
print(f"\nSample values:")
print(y1_risk_profile.head(10).tolist())

Target 1: Risk Profile (Classification)
Shape: (5000,)

Distribution:
risk_profile
Moderate        1901
Aggressive      1571
Conservative    1528
Name: count, dtype: int64

Sample values:
[np.str_('Moderate'), np.str_('Moderate'), np.str_('Moderate'), np.str_('Moderate'), np.str_('Aggressive'), np.str_('Aggressive'), np.str_('Aggressive'), np.str_('Moderate'), np.str_('Moderate'), np.str_('Conservative')]


In [22]:
# Target 2: Allocation (Regression - 2 outputs: equity%, bond%)
# Extract equity and bond percentages from allocation dict

def extract_allocation(allocation_dict):
    """Extract equity and bond percentages from allocation dictionary"""
    if isinstance(allocation_dict, dict):
        return allocation_dict.get('equity', 0.0), allocation_dict.get('bond', 0.0)
    else:
        # Handle string representation of dict
        import ast
        try:
            d = ast.literal_eval(str(allocation_dict))
            return d.get('equity', 0.0), d.get('bond', 0.0)
        except:
            return 0.0, 0.0

allocations = result_df['allocation'].apply(extract_allocation)
y2_allocation = pd.DataFrame(allocations.tolist(), columns=['equity_pct', 'bond_pct'])

print("Target 2: Allocation (Regression)")
print(f"Shape: {y2_allocation.shape}")
print(f"\nStatistics:")
print(y2_allocation.describe())
print(f"\nFirst 10 rows:")
print(y2_allocation.head(10))

Target 2: Allocation (Regression)
Shape: (5000, 2)

Statistics:
        equity_pct     bond_pct
count  5000.000000  5000.000000
mean      0.567491     0.332361
std       0.181956     0.182189
min       0.217555     0.009769
25%       0.384866     0.133876
50%       0.550005     0.350346
75%       0.764037     0.516057
max       0.935319     0.668150

First 10 rows:
   equity_pct  bond_pct
0    0.481960  0.440357
1    0.558489  0.312604
2    0.535996  0.361771
3    0.509447  0.377699
4    0.759111  0.129084
5    0.757414  0.144294
6    0.831857  0.079268
7    0.589695  0.313404
8    0.542628  0.338572
9    0.375224  0.534717


In [23]:
# Verify allocations sum to ~1.0 (with alternative)
sample_allocations = result_df['allocation'].head(5)
print("Sample allocations (first 5):")
for i, alloc in enumerate(sample_allocations):
    if isinstance(alloc, dict):
        total = alloc.get('equity', 0) + alloc.get('bond', 0) + alloc.get('alternative', 0)
        print(f"{i}: {alloc} → Sum: {total:.3f}")

Sample allocations (first 5):
0: {'equity': 0.4819597900977592, 'bond': 0.44035686697664933, 'alternative': 0.07768334292559141} → Sum: 1.000
1: {'equity': 0.55848927600329, 'bond': 0.3126036814300599, 'alternative': 0.12890704256665012} → Sum: 1.000
2: {'equity': 0.5359957294249246, 'bond': 0.36177070677965534, 'alternative': 0.10223356379541998} → Sum: 1.000
3: {'equity': 0.5094468539372864, 'bond': 0.37769887799829016, 'alternative': 0.11285426806442359} → Sum: 1.000
4: {'equity': 0.7591109903717875, 'bond': 0.12908376235028413, 'alternative': 0.11180524727792847} → Sum: 1.000


In [24]:
# Target 3: Total ETFs (Regression - integer)
y3_total_etfs = result_df['total_etfs'].copy()

print("Target 3: Total ETFs (Regression)")
print(f"Shape: {y3_total_etfs.shape}")
print(f"\nStatistics:")
print(y3_total_etfs.describe())
print(f"\nDistribution:")
print(y3_total_etfs.value_counts().sort_index())
print(f"\nFirst 10 values:")
print(y3_total_etfs.head(10).tolist())

Target 3: Total ETFs (Regression)
Shape: (5000,)

Statistics:
count    5000.000000
mean        9.808400
std         1.280632
min         7.000000
25%         9.000000
50%        10.000000
75%        11.000000
max        13.000000
Name: total_etfs, dtype: float64

Distribution:
total_etfs
7      121
8      690
9     1270
10    1403
11    1042
12     409
13      65
Name: count, dtype: int64

First 10 values:
[10, 10, 11, 11, 7, 8, 10, 9, 11, 11]


## Step 7: Feature Engineering - Identify Categorical vs Numerical

In [26]:
# Identify categorical and numerical features (using ACTUAL column names)
categorical_features = ['Gender', 'Education', 'MaritalStatus', 'InvestmentGoal', 'risk_tolerance', 'FinancialInvolvement']
numerical_features = ['age', 'HouseholdSize', 'income', 'horizon', 'InvestmentCapital', 'experience']

print("Categorical Features:")
for feat in categorical_features:
    print(f"  - {feat}: {X[feat].nunique()} unique values")
    print(f"    Values: {X[feat].unique().tolist()}")

print(f"\n{'='*60}")
print("Numerical Features:")
for feat in numerical_features:
    print(f"  - {feat}: Range [{X[feat].min()}, {X[feat].max()}]")

Categorical Features:
  - Gender: 2 unique values
    Values: ['Female', 'Male']
  - Education: 3 unique values
    Values: ['Master', 'High school', 'Bachelor']
  - MaritalStatus: 4 unique values
    Values: ['Married', 'Single', 'Divorced', 'Widowed']
  - InvestmentGoal: 4 unique values
    Values: ['Child education', 'Retirement', 'Home purchase', 'Other']
  - risk_tolerance: 3 unique values
    Values: ['High', 'Medium', 'Low']
  - FinancialInvolvement: 3 unique values
    Values: ['High', 'Medium', 'Low']

Numerical Features:
  - age: Range [18, 90]
  - HouseholdSize: Range [1, 4]
  - income: Range [14034, 215730]
  - horizon: Range [1, 39]
  - InvestmentCapital: Range [1400, 996445]
  - experience: Range [0, 68]


## Step 8: Encode and Scale Features

In [27]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

# Fit and transform features
X_processed = preprocessor.fit_transform(X)

# Get feature names after encoding
num_feature_names = numerical_features
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = num_feature_names + list(cat_feature_names)

# Convert to DataFrame
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)

print(f"✅ Features encoded and scaled")
print(f"\nOriginal features: {X.shape[1]}")
print(f"Processed features: {X_processed_df.shape[1]}")
print(f"\nFeature names after encoding:")
print(all_feature_names)
print(f"\n{'='*60}")
print("First 5 rows of processed features:")
X_processed_df.head()

✅ Features encoded and scaled

Original features: 12
Processed features: 19

Feature names after encoding:
['age', 'HouseholdSize', 'income', 'horizon', 'InvestmentCapital', 'experience', 'Gender_Male', 'Education_High school', 'Education_Master', 'MaritalStatus_Married', 'MaritalStatus_Single', 'MaritalStatus_Widowed', 'InvestmentGoal_Home purchase', 'InvestmentGoal_Other', 'InvestmentGoal_Retirement', 'risk_tolerance_Low', 'risk_tolerance_Medium', 'FinancialInvolvement_Low', 'FinancialInvolvement_Medium']

First 5 rows of processed features:


,age,HouseholdSize,income,horizon,InvestmentCapital,experience,Gender_Male,Education_High school,Education_Master,MaritalStatus_Married,MaritalStatus_Single,MaritalStatus_Widowed,InvestmentGoal_Home purchase,InvestmentGoal_Other,InvestmentGoal_Retirement,risk_tolerance_Low,risk_tolerance_Medium,FinancialInvolvement_Low,FinancialInvolvement_Medium
0,0.180694,0.553779,0.915761,-0.304866,0.366334,0.283553,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-0.253902,-0.418445,0.298345,-0.742609,-0.055671,-0.963451,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
2,-0.036604,1.526002,-0.597463,0.351748,-0.319176,0.673242,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,-0.416875,0.553779,0.098561,-0.523737,-0.019376,-0.729638,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
4,-1.394715,-0.418445,-1.021760,-1.180352,-0.567943,-0.963451,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


## Step 9: Train/Test Split

In [28]:
# Split with stratification on risk_profile to maintain class balance
X_train, X_test, y1_train, y1_test, y2_train, y2_test, y3_train, y3_test = train_test_split(
    X_processed_df,
    y1_risk_profile,
    y2_allocation,
    y3_total_etfs,
    test_size=0.2,
    random_state=42,
    stratify=y1_risk_profile  # Maintain risk profile balance
)

print("✅ Train/Test Split Complete (80/20)")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny1_train (risk_profile) shape: {y1_train.shape}")
print(f"y1_test (risk_profile) shape: {y1_test.shape}")
print(f"\ny2_train (allocation) shape: {y2_train.shape}")
print(f"y2_test (allocation) shape: {y2_test.shape}")
print(f"\ny3_train (total_etfs) shape: {y3_train.shape}")
print(f"y3_test (total_etfs) shape: {y3_test.shape}")

✅ Train/Test Split Complete (80/20)

X_train shape: (4000, 19)
X_test shape: (1000, 19)

y1_train (risk_profile) shape: (4000,)
y1_test (risk_profile) shape: (1000,)

y2_train (allocation) shape: (4000, 2)
y2_test (allocation) shape: (1000, 2)

y3_train (total_etfs) shape: (4000,)
y3_test (total_etfs) shape: (1000,)


In [29]:
# Verify stratification worked
print("Risk Profile Distribution in Train Set:")
print(y1_train.value_counts())
print("\nPercentages:")
print(y1_train.value_counts(normalize=True) * 100)

print(f"\n{'='*60}")
print("Risk Profile Distribution in Test Set:")
print(y1_test.value_counts())
print("\nPercentages:")
print(y1_test.value_counts(normalize=True) * 100)

Risk Profile Distribution in Train Set:
risk_profile
Moderate        1521
Aggressive      1257
Conservative    1222
Name: count, dtype: int64

Percentages:
risk_profile
Moderate        38.025
Aggressive      31.425
Conservative    30.550
Name: proportion, dtype: float64

Risk Profile Distribution in Test Set:
risk_profile
Moderate        380
Aggressive      314
Conservative    306
Name: count, dtype: int64

Percentages:
risk_profile
Moderate        38.0
Aggressive      31.4
Conservative    30.6
Name: proportion, dtype: float64


## Step 10: Save Prepared Data

In [31]:
# Create data directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Save all datasets
X_train.to_csv('data/X_train.csv', index=False)
X_test.to_csv('data/X_test.csv', index=False)

y1_train.to_csv('data/y_train_profile.csv', index=False, header=['risk_profile'])
y1_test.to_csv('data/y_test_profile.csv', index=False, header=['risk_profile'])

y2_train.to_csv('data/y_train_allocation.csv', index=False)
y2_test.to_csv('data/y_test_allocation.csv', index=False)

y3_train.to_csv('data/y_train_etfs.csv', index=False, header=['total_etfs'])
y3_test.to_csv('data/y_test_etfs.csv', index=False, header=['total_etfs'])

print("✅ All data saved to 'data/' directory")
print("\nFiles created:")
for file in os.listdir('data'):
    filepath = os.path.join('data', file)
    size = os.path.getsize(filepath)
    print(f"  - {file} ({size:,} bytes)")

✅ All data saved to 'data/' directory

Files created:
  - X_test.csv (172,343 bytes)
  - X_train.csv (687,920 bytes)
  - y_test_allocation.csv (39,479 bytes)
  - y_test_etfs.csv (3,583 bytes)
  - y_test_profile.csv (11,866 bytes)
  - y_train_allocation.csv (157,884 bytes)
  - y_train_etfs.csv (14,360 bytes)
  - y_train_profile.csv (47,416 bytes)


In [32]:
# Save the preprocessor for later use during inference
import joblib

joblib.dump(preprocessor, 'data/preprocessor.pkl')
print("✅ Preprocessor saved to 'data/preprocessor.pkl'")

✅ Preprocessor saved to 'data/preprocessor.pkl'


## Step 11: Summary Statistics

In [33]:
print("="*60)
print("PHASE 2 COMPLETE: DATA PREPARATION SUMMARY")
print("="*60)
print(f"\n📊 Dataset Sizes:")
print(f"  Total samples: {len(X_processed_df)}")
print(f"  Training samples: {len(X_train)} (80%)")
print(f"  Test samples: {len(X_test)} (20%)")

print(f"\n🎯 Targets:")
print(f"  Target 1 (Risk Profile): 3 classes - {y1_train.unique().tolist()}")
print(f"  Target 2 (Allocation): 2 continuous outputs - equity_pct, bond_pct")
print(f"  Target 3 (Total ETFs): Integer range [{y3_train.min()}, {y3_train.max()}]")

print(f"\n📁 Files Saved:")
print(f"  - X_train.csv, X_test.csv")
print(f"  - y_train_profile.csv, y_test_profile.csv")
print(f"  - y_train_allocation.csv, y_test_allocation.csv")
print(f"  - y_train_etfs.csv, y_test_etfs.csv")
print(f"  - preprocessor.pkl")

print(f"\n✅ Ready for Phase 3: Model Training!")
print("="*60)

PHASE 2 COMPLETE: DATA PREPARATION SUMMARY

📊 Dataset Sizes:
  Total samples: 5000
  Training samples: 4000 (80%)
  Test samples: 1000 (20%)

🎯 Targets:
  Target 1 (Risk Profile): 3 classes - [np.str_('Aggressive'), np.str_('Moderate'), np.str_('Conservative')]
  Target 2 (Allocation): 2 continuous outputs - equity_pct, bond_pct
  Target 3 (Total ETFs): Integer range [7, 13]

📁 Files Saved:
  - X_train.csv, X_test.csv
  - y_train_profile.csv, y_test_profile.csv
  - y_train_allocation.csv, y_test_allocation.csv
  - y_train_etfs.csv, y_test_etfs.csv
  - preprocessor.pkl

✅ Ready for Phase 3: Model Training!
